## Plots for results

In [ ]:
import os # thesis-segmentation ENV
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.patches import Patch
import textwrap
import pickle

from skimage.transform import resize, AffineTransform, warp

from tifffile import imread

from __future__ import print_function

In [ ]:
import sys
sys.path.insert(0, r"D:\Masters_Medical_Informatics_Larger_Files\Thesis\Working_Folder\Masters-Thesis\src\he2multi_reg")

In [ ]:
import importlib, he2multi_reg
importlib.reload(he2multi_reg)
importlib.reload(he2multi_reg.reg)
importlib.reload(he2multi_reg.metrics)
importlib.reload(he2multi_reg.regPipeline)
importlib.reload(he2multi_reg.preprocess)

In [ ]:
def load_image_data(folder_path):

    img_data = []
    label_data = []
    
    for _, _, files in os.walk(folder_path):
        for index, file in enumerate(files):
            if file.endswith(".tif"): 
                image_path = os.path.join(folder_path, file)
                img_raw = imread(image_path)
                img = np.array(img_raw)
                img_data.append(img)

    return img_data

#### cropped images

In [ ]:
hne_px_sz = 0.5023
dapi_px_sz = 0.209877

scale = hne_px_sz / dapi_px_sz

# load dapi
dapi_img_data = load_image_data("../../../Data/for_testing/exmp_1/cropped/dapi")
dapi1_init = resize(dapi_img_data[0], (int(dapi_img_data[0].shape[0]/scale), int(dapi_img_data[0].shape[1]/scale)), anti_aliasing=True)
dapi2_init = resize(dapi_img_data[1], (int(dapi_img_data[1].shape[0]/scale), int(dapi_img_data[1].shape[1]/scale)), anti_aliasing=True)
dapi1_init, dapi2_init = dapi1_init*255, dapi2_init*255

# load hne
hne_image_data = load_image_data("../../../Data/for_testing/exmp_1/cropped/hne")
hne1_init = hne_image_data[0]
hne2_init = hne_image_data[1]

hne1_deconv = he2multi_reg.colour_deconvolusion_preprocessing_HnE(hne1_init)
hne2_deconv = he2multi_reg.colour_deconvolusion_preprocessing_HnE(hne2_init)

In [ ]:
def overlay_images(dapi_img, hne_img, axis, alpha_dapi=1, alpha_hne=0.35):
    axis.imshow(dapi_img, cmap='Greens', alpha=alpha_dapi)
    axis.imshow(hne_img, cmap='Reds', alpha=alpha_hne)
    axis.set_aspect('equal')
    axis.set_xticks([])
    axis.set_yticks([])

def get_nice_ticks(max_val, n_ticks=5):
    step = max_val // n_ticks
    step = int(np.ceil(step / 50) * 50)  # round to nearest 50
    ticks = np.arange(0, step * n_ticks, step)  # exclude the last value to avoid overlap
    return ticks

def plot_comparison_grid(
        dapi_list,
        hne_list,
        deconv_list,
        registered_list_dict,
        column_titles=None
    ):
    n_rows = len(dapi_list)
    reg_methods = list(registered_list_dict.keys())
    n_cols = 3 + len(reg_methods)

    # set equal image sizes
    min_shape = np.min([img.shape[:2] for img in dapi_list], axis=0)
    def resize_to_min_shape(img_list, min_shape):
        return [img[:min_shape[0], :min_shape[1]] for img in img_list]

    dapi_list = resize_to_min_shape(dapi_list, min_shape)
    hne_list = resize_to_min_shape(hne_list, min_shape)
    deconv_list = resize_to_min_shape(deconv_list, min_shape)
    for method in reg_methods:
        registered_list_dict[method] = resize_to_min_shape(registered_list_dict[method], min_shape)

    if column_titles is None:
        column_titles = ["DAPI", "H&E", "initial overlay"] + reg_methods

    fig, axes = plt.subplots(
        n_rows + 1,
        n_cols,
        figsize=(1.5 * n_cols, 2.6 * (n_rows + 0.2)),
        squeeze=False
    )

    # top row titles
    for j in range(n_cols):
        ax = axes[0, j]
        title = column_titles[j]
        wrapped_title = "\n".join(textwrap.wrap(title, width=15))
        ax.set_title(wrapped_title, fontsize=12, pad=3)
        ax.axis("off")

    # image rows
    n_ticks = 5

    # image rows
    for i in range(n_rows):
        for j in range(n_cols):
            ax = axes[i + 1, j]
        
            # which image to plot
            if j == 0:
                img = dapi_list[i]
            elif j == 1:
                img = hne_list[i]
            elif j == 2:
                img = deconv_list[i]
            else:
                method = reg_methods[j-3]
                img = registered_list_dict[method][i]
        
            # overlay 
            if j >= 2:
                overlay_images(dapi_list[i], img, ax)
            else:
                ax.imshow(img, cmap="gray")
                ax.set_aspect('equal')
        
            # show only for left column (y) and bottom row (x)
            if j == 0:  # left column
                yticks = get_nice_ticks(min_shape[0], n_ticks=5)
                ax.set_yticks(yticks)
                ax.set_yticklabels([f"{int(y)}" for y in yticks], fontsize=8)
            else:
                ax.set_yticks([])

            if i == n_rows - 1:  # bottom row
                xticks = get_nice_ticks(min_shape[1], n_ticks=5)
                ax.set_xticks(xticks)
                ax.set_xticklabels([f"{int(x)}" for x in xticks], fontsize=8)
            else:
                ax.set_xticks([])


    plt.subplots_adjust(
        left=0.03, right=0.97,
        top=0.95, bottom=0.05,
        wspace=0.02,
        hspace=0.01
    )

    plt.show()

In [ ]:
img1_tforms, img1_registered, img1_tre_pts = he2multi_reg.register_DAPI_HnE(dapi1_init, hne1_deconv, adv_tform='intensity', intensity_tform='r-af-bs', mpp=hne_px_sz)

In [ ]:
img2_tforms, img2_registered, img2_tre_pts = he2multi_reg.register_DAPI_HnE(dapi2_init, hne2_deconv, adv_tform='intensity', intensity_tform='r-af-bs', mpp=hne_px_sz)

In [ ]:
hne1_applying_tform = AffineTransform(scale=(1.0, 1.0), rotation=0.5, translation=(100, -200))
hne1_testing_hne_img = warp(hne1_init, hne1_applying_tform)
hne1_testing_img = warp(hne1_deconv, hne1_applying_tform)
img1_test_tforms, img1_test_registered, img1_test_tre_pts = he2multi_reg.register_DAPI_HnE(dapi1_init, hne1_testing_img, adv_tform='intensity', intensity_tform='r-af-bs', mpp=hne_px_sz)

In [ ]:
plot_comparison_grid(
    dapi_list=[dapi1_init, dapi2_init, dapi1_init],
    hne_list=[hne1_init, hne2_init, hne1_testing_hne_img],
    deconv_list=[hne1_deconv, hne2_deconv, hne1_testing_img],
    registered_list_dict={
        "rigid (feature-based)": [img1_registered['initial similarity'], img2_registered['initial similarity'], img1_test_registered['initial similarity']],
        "rigid (intensity-based)": [img1_registered['intensity based']['rigid'], img2_registered['intensity based']['rigid'], img1_test_registered['intensity based']['rigid']],
        "affine (intensity-based)": [img1_registered['intensity based']['affine'], img2_registered['intensity based']['affine'], img1_test_registered['intensity based']['affine']],
        "bspline (intensity-based)": [img1_registered['intensity based']['bspline'], img2_registered['intensity based']['bspline'], img1_test_registered['intensity based']['bspline']]
    }
)

#### ROI images

In [ ]:
import sys
sys.path.insert(0, "/home-link/zxovq55/Masters-Thesis/src/registration")

In [ ]:
import matplotlib.pyplot as plt
from matplotlib.patches import Patch
from tifffile import imread
from skimage.transform import resize
import numpy as np
import textwrap

import importlib, registration
importlib.reload(registration)
importlib.reload(registration.reg)
importlib.reload(registration.metrics)
importlib.reload(registration.regPipeline)
importlib.reload(registration.preprocess)

In [ ]:
hne_px_sz = 0.5023
dapi_px_sz = 0.209877

scale = hne_px_sz / dapi_px_sz

# load dapi
dapi_1_data = np.array(imread("../../data/dapi_1.tif"))
dapi1_init = resize(dapi_1_data, (int(dapi_1_data.shape[0]/scale), int(dapi_1_data.shape[1]/scale)), anti_aliasing=True)

dapi_2_data = np.array(imread("../../data/dapi_2.tif"))
dapi2_init = resize(dapi_2_data, (int(dapi_2_data.shape[0]/scale), int(dapi_2_data.shape[1]/scale)), anti_aliasing=True)

dapi_3_data = np.array(imread("../../data/dapi_3.tif"))
dapi3_init = resize(dapi_3_data, (int(dapi_3_data.shape[0]/scale), int(dapi_3_data.shape[1]/scale)), anti_aliasing=True)

dapi1_init, dapi2_init, dapi3_init = dapi1_init*255, dapi2_init*255, dapi3_init*255

# load hne
hne_1_init = np.array(imread("../../data/hne_1.tif"))
hne_2_init = np.array(imread("../../data/hne_2.tif"))
hne_3_init = np.array(imread("../../data/hne_3.tif"))

# preprocess hne
hne1_deconv = registration.colour_deconvolusion_preprocessing_HnE(hne_1_init)
hne2_deconv = registration.colour_deconvolusion_preprocessing_HnE(hne_2_init)
hne3_deconv = registration.colour_deconvolusion_preprocessing_HnE(hne_3_init)

In [ ]:
def overlay_images(dapi_img, hne_img, axis, alpha_dapi=1, alpha_hne=0.35):
    axis.imshow(dapi_img, cmap='Greens', alpha=alpha_dapi)
    axis.imshow(hne_img, cmap='Reds', alpha=alpha_hne)
    axis.set_aspect('equal')
    axis.set_xticks([])
    axis.set_yticks([])

def get_nice_ticks(max_val, n_ticks=5):
    step = max_val // n_ticks
    step = int(np.ceil(step / 50) * 50)  # round to nearest 50
    ticks = np.arange(0, step * n_ticks, step)  # exclude the last value to avoid overlap
    return ticks

def plot_comparison_grid2(
        dapi_list,
        hne_list,
        deconv_list,
        registered_list_dict,
        column_titles=None
    ):
    n_rows = len(dapi_list)
    reg_methods = list(registered_list_dict.keys())
    n_cols = 3 + len(reg_methods)

    # set equal image sizes
    min_shape = np.min([img.shape[:2] for img in dapi_list], axis=0)
    def resize_to_min_shape(img_list, min_shape):
        return [img[:min_shape[0], :min_shape[1]] for img in img_list]

    dapi_list = resize_to_min_shape(dapi_list, min_shape)
    hne_list = resize_to_min_shape(hne_list, min_shape)
    deconv_list = resize_to_min_shape(deconv_list, min_shape)
    for method in reg_methods:
        registered_list_dict[method] = resize_to_min_shape(registered_list_dict[method], min_shape)

    if column_titles is None:
        column_titles = ["DAPI", "H&E", "initial overlay"] + reg_methods

    fig, axes = plt.subplots(
        n_rows + 1,
        n_cols,
        figsize=(1.5 * n_cols, 1.65 * (n_rows + 0.2)),
        squeeze=False
    )

    # top row titles
    for j in range(n_cols):
        ax = axes[0, j]
        title = column_titles[j]
        wrapped_title = "\n".join(textwrap.wrap(title, width=15))
        ax.set_title(wrapped_title, fontsize=12, pad=3)
        ax.axis("off")


    # image rows
    for i in range(n_rows):
        for j in range(n_cols):
            ax = axes[i + 1, j]
        
            # which image to plot
            if j == 0:
                img = dapi_list[i]
            elif j == 1:
                img = hne_list[i]
            elif j == 2:
                img = deconv_list[i]
            else:
                method = reg_methods[j-3]
                img = registered_list_dict[method][i]
        
            # overlay 
            if j >= 2:
                overlay_images(dapi_list[i], img, ax)
            else:
                ax.imshow(img, cmap="gray")
                ax.set_aspect('equal')
        
            # show only for left column (y) and bottom row (x)
            if j == 0:  # left column
                yticks = get_nice_ticks(min_shape[0], n_ticks=4)
                ax.set_yticks(yticks)
                ax.set_yticklabels([f"{int(y)}" for y in yticks], fontsize=8)
            else:
                ax.set_yticks([])

            if i == n_rows - 1:  # bottom row
                xticks = get_nice_ticks(min_shape[1], n_ticks=4)
                ax.set_xticks(xticks)
                ax.set_xticklabels([f"{int(x)}" for x in xticks], fontsize=8)
            else:
                ax.set_xticks([])


    plt.subplots_adjust(
        left=0.03, right=0.97,
        top=0.95, bottom=0.05,
        wspace=0.02,
        hspace=0.01
    )

    plt.show()

In [ ]:
img1_tforms, img1_registered, img1_tre_pts = registration.register_DAPI_HnE(dapi1_init, hne1_deconv, adv_tform='intensity', intensity_tform='r-af-bs', mpp=hne_px_sz)

In [ ]:
img2_tforms, img2_registered, img2_tre_pts = registration.register_DAPI_HnE(dapi2_init, hne2_deconv, adv_tform='intensity', intensity_tform='r-af-bs', mpp=hne_px_sz)

In [ ]:
img3_tforms, img3_registered, img3_tre_pts = registration.register_DAPI_HnE(dapi3_init, hne3_deconv, adv_tform='intensity', intensity_tform='r-af-bs', mpp=hne_px_sz)

In [ ]:
plot_comparison_grid2(
    dapi_list=[dapi1_init[:,1000:], np.rot90(dapi2_init, k=1), dapi3_init],
    hne_list=[hne_1_init[:,1000:], np.rot90(hne_2_init, k=1), hne_3_init],
    deconv_list=[hne1_deconv[:,1000:], np.rot90(hne2_deconv, k=1), hne3_deconv],
    registered_list_dict={
        "rigid (feature-based)": [img1_registered['initial similarity'][:,1000:], np.rot90(img2_registered['initial similarity'], k=1), img3_registered['initial similarity']],
        "rigid (intensity-based)": [img1_registered['intensity based']['rigid'][:,1000:], np.rot90(img2_registered['intensity based']['rigid'], k=1), img3_registered['intensity based']['rigid']],
        "affine (intensity-based)": [img1_registered['intensity based']['affine'][:,1000:], np.rot90(img2_registered['intensity based']['affine'], k=1), img3_registered['intensity based']['affine']],
        "bspline (intensity-based)": [img1_registered['intensity based']['bspline'][:,1000:], np.rot90(img2_registered['intensity based']['bspline'], k=1), img3_registered['intensity based']['bspline']]
    }
)

#### plots

##### Comparison between methods

In [ ]:
# Boxplot for different registration methods
# rTRE

num_imgs = 3
tre_metrics = []

for file_tag in ['ftr', 'r', 'af', 'bs']:
    metrics_list = []
    
    for img_idx in range(num_imgs):

        with open(f"result_plots_metrics/{img_idx}_tre_{file_tag}_dict.pkl", "rb") as f:
            tre_dict = pickle.load(f)

            metrics_list = np.concatenate((metrics_list, tre_dict[(f"img_{img_idx+1}")]), axis=0)
            metrics_list = metrics_list.ravel()


    tre_metrics.append(metrics_list)

methods = ["rigid\n(feature-based)", "rigid\n(intensity-based)", "affine\n(intensity-based)", "bspline\n(intensity-based)"]

fig, ax = plt.subplots(figsize=(8,6))

box = ax.boxplot(tre_metrics, tick_labels=methods, patch_artist=True, showfliers=True, 
                 capprops=dict(color='none'), 
                 whiskerprops=dict(color='black'), 
                 boxprops=dict(color='black'), 
                 medianprops=dict(color='black', linewidth=2),
                 flierprops=dict(marker='o', markersize=6, markerfacecolor='black'))

colors = ["lightblue", "lightgreen", "lightpink", "lightyellow"]
for patch, color in zip(box['boxes'], colors):
    patch.set_facecolor(color)

y_min = min(np.min(arr) for arr in tre_metrics)
y_max = max(np.max(arr) for arr in tre_metrics)

ax.set_ylim(y_min-0.00005, y_max+0.0003)


ax.set_ylabel("rTRE Metric")
ax.set_title("Comparison of Registration Methods")
plt.show()


In [ ]:
# Boxplot for different registration methods
# MI

num_imgs = 3
mi_metrics = []

for file_tag in ['ftr', 'r', 'af', 'bs']:
    metrics_list = []
    
    for img_idx in range(num_imgs):

        with open(f"result_plots_metrics/{img_idx}_mi_{file_tag}_dict.pkl", "rb") as f:
            mi_dict = pickle.load(f)

            metrics_list = np.concatenate((metrics_list, mi_dict[(f"img_{img_idx+1}")]), axis=0)
            metrics_list = metrics_list.ravel()


    mi_metrics.append(metrics_list)
    
methods = ["rigid\n(feature-based)", "rigid\n(intensity-based)", "affine\n(intensity-based)", "bspline\n(intensity-based)"]

fig, ax = plt.subplots(figsize=(8,6))

box = ax.boxplot(mi_metrics, tick_labels=methods, patch_artist=True, showfliers=True, 
                 capprops=dict(color='none'), 
                 whiskerprops=dict(color='black'), 
                 boxprops=dict(color='black'), 
                 medianprops=dict(color='black', linewidth=2),
                 flierprops=dict(marker='o', markersize=6, markerfacecolor='black'))

# Color the boxes
colors = ["lightblue", "lightgreen", "lightpink", "lightyellow"]
for patch, color in zip(box['boxes'], colors):
    patch.set_facecolor(color)

y_min = min(np.min(arr) for arr in mi_metrics)
y_max = max(np.max(arr) for arr in mi_metrics)

ax.set_ylim(y_min-0.05, y_max+0.05)

ax.set_ylabel("NMI Metric")
ax.set_title("Comparison of Registration Methods")
plt.show()


In [ ]:
# Scatter plot comparing two methods 
# tre

ftr_r = tre_metrics[0] # rigid (feature-based)
intns = tre_metrics[1] # rigid (intensity-based)

fig, ax = plt.subplots(figsize=(6,6))

ax.scatter(ftr_r, intns, c='black', s=60, alpha=0.8, edgecolor='k')
min_val = min(ftr_r.min(), intns.min())
max_val = max(ftr_r.max(), intns.max())
ax.plot([min_val, max_val], [min_val, max_val], 'r--', linewidth=1.5)

ax.set_xlabel("rigid (feature-based)")
ax.set_ylabel("rigid (intensity-based)")
ax.set_title("Pairwise Comparison of Two Methods with rTRE")
ax.set_aspect('equal', adjustable='box')  


plt.show()

In [ ]:
# Scatter plot comparing two methods 
# mi

ftr_r = mi_metrics[0] # rigid (feature-based)
intns = mi_metrics[1] # rigid (intensity-based)

fig, ax = plt.subplots(figsize=(6,6))

ax.scatter(ftr_r, intns, c='black', s=60, alpha=0.8, edgecolor='k')
min_val = min(ftr_r.min(), intns.min())
max_val = max(ftr_r.max(), intns.max())
ax.plot([min_val, max_val], [min_val, max_val], 'r--', linewidth=1.5)

ax.invert_yaxis()
ax.invert_xaxis()
ax.set_xlabel("rigid (feature-based)")
ax.set_ylabel("rigid (intensity-based)")
ax.set_title("Pairwise Comparison of Two Methods with NMI")
ax.set_aspect('equal', adjustable='box')  


plt.show()

##### Run time comparison

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

# Example data for 10 runs
runs = np.arange(1, 11)  # X-axis points (runs)
time_first_registration = np.array([10, 12, 9, 11, 13, 12, 10, 14, 11, 12])
time_next_registration = np.array([5, 6, 4, 5, 6, 5, 4, 6, 5, 5])
time_metrics = np.array([3, 4, 2, 3, 3, 4, 3, 2, 3, 3])

# Stack the data
stacked_data = np.vstack([time_first_registration,
                          time_next_registration,
                          time_metrics])

# Create stacked area plot
plt.stackplot(runs, stacked_data, labels=["First Registration", "Next Registration", "Metrics"], alpha=0.7)

# Remove x-axis ticks
plt.xticks([])

# Labels
plt.ylabel("Time (seconds)")
plt.title("Runtime Breakdown per Run")
plt.legend(loc='upper left')
plt.show()
